# nb_01_bronze: land the four raw sources exactly as they arrived
**Task 1.** Everything is read as a string. No cleaning, no casting. Only two lineage columns are added: `ingested_at` and `source_file`.

Before running: attach the Lakehouse `lh_buildmate` (Explorer pane, Add Lakehouse) and set `BASE` below to the folder you see under Files.

In [ ]:
from pyspark.sql import functions as F

# Folder that holds customer_master/, rentals/, billing/, depots/ . Change it if your Files tree differs.
BASE = "Files/raw"            # or "Files/buildmate_raw_data/raw"

In [ ]:
def land(path, table):
    """Read a CSV source with every column as string, add lineage, write a Bronze Delta table."""
    df = (spark.read
          .option("header", True)
          .option("inferSchema", False)          # keep every column a string: Bronze is a photograph
          .csv(path)
          .withColumn("ingested_at", F.current_timestamp())
          .withColumn("source_file", F.col("_metadata.file_path")))   # F.input_file_name() also works
    (df.write.mode("overwrite").format("delta")
       .option("overwriteSchema", "true").saveAsTable(table))
    print(f"{table:18s} rows = {spark.table(table).count()}")

land(f"{BASE}/customer_master/customer_master_export.csv", "bronze_customers")
land(f"{BASE}/depots/depot_master.csv",                    "bronze_depots")
land(f"{BASE}/rentals/rentals_*.csv",                      "bronze_rentals")   # ONE wildcard load, all 31 files
land(f"{BASE}/billing/billing_export.csv",                 "bronze_billing")

## Prove the re-sent day (10 June) using the lineage column

In [ ]:
r = (spark.table("bronze_rentals")
     .withColumn("file", F.regexp_extract("source_file", r"([^/]+\.csv)$", 1)))

jun10 = r.filter(F.col("file").startswith("rentals_2026-06-10"))

print("Rows per source file for 10 June (two files, same row count):")
display(jun10.groupBy("file")
             .agg(F.count("*").alias("rows"),
                  F.countDistinct("rental_id").alias("distinct_rental_ids")))

main   = jun10.filter(~F.col("file").contains("resend")).select("rental_id")
resend = jun10.filter(F.col("file").contains("resend")).select("rental_id")
print("rental_ids that appear in BOTH files:", main.intersect(resend).count())

## Acceptance checks (screenshot this cell's output)

In [ ]:
expected = {"bronze_customers": 602, "bronze_depots": 6, "bronze_rentals": 976, "bronze_billing": 787}
for t, n in expected.items():
    df = spark.table(t)
    assert df.count() == n, f"{t}: expected {n}, got {df.count()}"
    source_cols = [f for f in df.schema.fields if f.name not in ("ingested_at", "source_file")]
    assert all(f.dataType.simpleString() == "string" for f in source_cols), f"{t}: a source column is not a string"
    assert {"ingested_at", "source_file"} <= set(df.columns), f"{t}: lineage columns missing"
    print(f"OK  {t:18s} rows={n:4d}  all source columns are string  lineage columns present")

n_files = spark.table("bronze_rentals").select("source_file").distinct().count()
assert n_files == 31, n_files
print("OK  bronze_rentals came from", n_files, "files")
spark.table("bronze_rentals").printSchema()